# Pipeline

For your RAG system, the pipeline is:

Document Loading → Chunking → Embedding → Vector DB → Retrieval → LLM Answer

### Why do we use pipeline:
- Modular architecture
- Each step is independent
- Easy to replace components (e.g., Pinecone → FAISS)

In [31]:
# import Libraries

import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from pinecone import Pinecone, ServerlessSpec

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain.schema import Document

In [32]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")

INDEX_NAME="langchainvector"

# Step 1: Create Classes for Each Stage

In [33]:
# 1. Document Loader

class DocumentLoader:
    def __init__(self, directory):
        self.directory = directory

    def load(self):
        loader = PyPDFDirectoryLoader(self.directory)
        return loader.load()


In [34]:
# 2. Chunking Stage
class DocumentChunker:
    def __init__(self, chunk_size=800, overlap=100):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap
        )

    def split(self, docs):
        chunked_docs = []

        for doc in docs:
            chunks = self.splitter.split_text(doc.page_content)

            for chunk in chunks:
                chunked_docs.append({
                    "text": chunk,
                    "metadata": doc.metadata
                })

        return chunked_docs


In [35]:
# 3. Embedding + Pinecone Store
class VectorStore:
    def __init__(self, api_key, index_name):
        self.pc = Pinecone(api_key=api_key)
        self.index_name = index_name

        if index_name not in self.pc.list_indexes().names():
            self.pc.create_index(
                name=index_name,
                dimension=1536,
                metric='euclidean',
                spec=ServerlessSpec(cloud='aws', region='us-east-1')
            )

        self.index = self.pc.Index(index_name)

        self.embeddings = OpenAIEmbeddings(
            model="text-embedding-3-small",
            api_key=OPENAI_API_KEY
        )

    def embed(self, text):
        return self.embeddings.embed_query(text)

    def store(self, documents):
        for i, doc in enumerate(documents):
            embedding = self.embed(doc["text"])

            self.index.upsert([{
                "id": str(i),
                "values": embedding,
                "metadata": {
                    "text": doc["text"],
                    "source": doc["metadata"].get("source"),
                    "page": doc["metadata"].get("page"),
                    "chunk_id": i
                }
            }])


In [36]:
# 4. Retriever
class Retriever:
    def __init__(self, vector_store):
        self.vector_store = vector_store

    def retrieve(self, query, k=2):
        query_embedding = self.vector_store.embed(query)

        results = self.vector_store.index.query(
            vector=query_embedding,
            top_k=k,
            include_metadata=True
        )

        docs = []
        for match in results['matches']:
            docs.append({
                "page_content": match['metadata']['text'],
                "metadata": match['metadata']
            })

        return docs


In [37]:
# 5. LLM Answer Generator
class QAChain:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

        self.prompt = ChatPromptTemplate.from_template(
            """
            Answer ONLY from the context.
            If the answer is not in the context, say "I don't know".

            Context:
            {context}

            Question:
            {user_query}
            """
        )

        self.chain = create_stuff_documents_chain(self.llm, self.prompt)

    def answer(self, query, retrieved_docs):
        docs = [Document(page_content=d["page_content"]) for d in retrieved_docs]

        response = self.chain.invoke({
            "user_query": query,
            "context": docs
        })

        return response


# Step 2: Final Pipeline Orchestrator


In [38]:
class RAGPipeline:
    def __init__(self, data_dir):
        self.loader = DocumentLoader(data_dir)
        self.chunker = DocumentChunker()
        self.vector_store = VectorStore(PINECONE_API_KEY, INDEX_NAME)
        self.retriever = Retriever(self.vector_store)
        self.qa_chain = QAChain()

    def ingest(self):
        docs = self.loader.load()
        chunks = self.chunker.split(docs)
        self.vector_store.store(chunks)
        print("Data ingested successfully")

    def query(self, question):
        retrieved_docs = self.retriever.retrieve(question)
        answer = self.qa_chain.answer(question, retrieved_docs)

        return answer, retrieved_docs


# Step3: Use the Pipeline

In [39]:
# Took 20 seconds
pipeline = RAGPipeline("documents-0-test")

# Step 1: Load + Store
pipeline.ingest()

# Step 2: Ask Question
answer, docs = pipeline.query("Who are the founders of Loxford Academy?")

print("Answer:", answer)

Data ingested successfully
Answer: The founders of Loxford Academy are Alberto Graciaso Negurasa and Rohit Salvadore Lomax.


## (OPTIONAL) If you  want to see the sources:

In [42]:
print("Sources: ")
for d in  docs:
    print("source: ", d['metadata']['source'])
    print("page:", d['metadata']['page'])
    print("chunk_id:", d['metadata']['chunk_id'])
    print("----------")

Sources: 
source:  documents-0-test\Quantum Mechanics-LoxfordAcademy-small.pdf
page: 1
chunk_id: 3
----------
source:  documents-0-test\Quantum Mechanics-LoxfordAcademy-small.pdf
page: 0
chunk_id: 1
----------


In [44]:
[i.name for i in pc.list_indexes()]

[]

# FINISH: Delete the index

In [43]:
from pinecone import Pinecone
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

# Delete
if INDEX_NAME in [i.name for i in pc.list_indexes()]:
    pc.delete_index(INDEX_NAME)
    print(f"Index '{INDEX_NAME}' deleted successfully")
else:
    print("Index not found")

Index not found
